# Tutorial — Antares Legacy Study to GEMS Converter

This tutorial shows how to convert a hand-built **Antares legacy study** into a runnable
GEMS study using the `antares-legacy-gems-converter` package and run the optimisation
with [GemsPy](https://gemspy.readthedocs.io/en/latest/).

**Study topology**

| Area | Components | Constant load |
|------|-----------|---------------|
| **north** | `gas_north` — 1 unit × 100 MW, 45 €/MWh<br>`wind` — 60 MW, sinusoidal profile | 120 MW |
| **south** | `gas_south` — 1 unit × 150 MW, 60 €/MWh | 80 MW |

Horizon: **168 hourly time steps** (one week), **1 scenario**.  
ENS cost: **20 000 €/MWh** — any unmet demand appears as costly unsupplied energy rather
than an infeasible solve.

**Workflow**

1. Build the Antares legacy study in-memory with `antares-craft`
2. Convert it to GEMS format with `AntaresStudyConverter`
3. Inspect the generated `system.yml`
4. Solve the optimisation with GemsPy
5. Visualise dispatch results per area

## Installation

In [ ]:
import subprocess, os

# Sync the doc dependency group (GemsPy + dev tools)
subprocess.run(
    ["uv", "sync", "--group", "doc", "--frozen"],
    check=True,
    capture_output=True,
    cwd=os.path.abspath(os.path.join(os.getcwd(), "../../..")),
)

# Install the Antares-to-GEMS converter
subprocess.run(
    ["uv", "pip", "install", "antares-legacy-gems-converter"],
    check=True,
)

## 1. Build the Antares Legacy Study

We use `antares-craft` to build a minimal two-area Antares study entirely in memory.
All timeseries are generated with NumPy — no input files on disk are required.

```
north ── gas_north  (1 × 100 MW, 45 €/MWh)    load 120 MW (constant)
      ── wind       (1 × 60 MW,  sinusoidal)

south ── gas_south  (1 × 150 MW, 60 €/MWh)    load  80 MW (constant)
```

> A north ↔ south link (50 MW each direction) is provided but commented out.
> Uncomment it to allow cross-area power exchange.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

# --- Working directories ---
_cwd = Path.cwd()
tmp_dir = _cwd / "tmp"                    # antares-craft writes the study here
gems_output_dir = _cwd / "tmp" / "gems"   # converter writes the GEMS study here

tmp_dir.mkdir(parents=True, exist_ok=True)
gems_output_dir.mkdir(parents=True, exist_ok=True)

print(f"Antares study parent : {tmp_dir}")
print(f"GEMS output dir      : {gems_output_dir}")

In [ ]:
from antares.craft import (
    create_study_local,
    AreaProperties,
    ThermalClusterProperties,
    RenewableClusterProperties,
    RenewableClusterGroup,
)

print("BUILDING ANTARES LEGACY STUDY")

study = create_study_local("legacy_tutorial", "920", tmp_dir)

# --- Areas ---
area_north = study.create_area(
    "north",
    properties=AreaProperties(energy_cost_unsupplied=20_000, energy_cost_spilled=1),
)
area_south = study.create_area(
    "south",
    properties=AreaProperties(energy_cost_unsupplied=20_000, energy_cost_spilled=1),
)
print("\t\u2713 Areas created: north, south")

# --- Loads (8760 rows; only the first 168 are used for the 1-week run) ---
area_north.set_load(pd.DataFrame(np.full((8760, 1), 120.0)))
area_south.set_load(pd.DataFrame(np.full((8760, 1), 80.0)))
print("\t\u2713 Loads set  : north = 120 MW, south = 80 MW (constant)")

# --- Thermal cluster: gas_north (1 unit × 100 MW, 45 \u20ac/MWh) ---
gas_north = area_north.create_thermal_cluster(
    "gas_north",
    ThermalClusterProperties(
        unit_count=1,
        nominal_capacity=100,
        marginal_cost=45,
        market_bid_cost=45,
    ),
)
gas_north.set_series(pd.DataFrame(np.ones((8760, 1))))

# --- Thermal cluster: gas_south (1 unit × 150 MW, 60 \u20ac/MWh) ---
gas_south = area_south.create_thermal_cluster(
    "gas_south",
    ThermalClusterProperties(
        unit_count=1,
        nominal_capacity=150,
        marginal_cost=60,
        market_bid_cost=60,
    ),
)
gas_south.set_series(pd.DataFrame(np.ones((8760, 1))))
print("\t\u2713 Thermal clusters: gas_north (100 MW, 45 \u20ac/MWh), gas_south (150 MW, 60 \u20ac/MWh)")

# --- Wind renewable cluster on north: sinusoidal capacity factor (0\u20131, 24-h period) ---
hours = np.arange(8760)
wind = area_north.create_renewable_cluster(
    "wind",
    RenewableClusterProperties(
        unit_count=1,
        nominal_capacity=60,
        group=RenewableClusterGroup.WIND_ON_SHORE,
    ),
)
wind.set_series(pd.DataFrame(0.5 + 0.5 * np.sin(2 * np.pi * hours / 24)))
print("\t\u2713 Wind cluster : 60 MW, sinusoidal profile (capacity factor 0\u20131)")

# Optional link \u2014 uncomment to allow power exchange between the two areas
# from antares.craft import LinkProperties
# link = study.create_link(area_from="north", area_to="south")
# link.set_capacity_direct(pd.DataFrame(np.full((8760, 1), 50.0)))
# link.set_capacity_indirect(pd.DataFrame(np.full((8760, 1), 50.0)))
# print("\t\u2713 Link created : north \u2194 south (50 MW each direction)")

print("\nAntares legacy study built successfully")

## 2. Convert to GEMS Format

`AntaresStudyConverter` reads the `antares-craft` study object and writes a GEMS-compatible
study under `gems_output_dir/input/`:

| Output path | Contents |
|-------------|----------|
| `input/system.yml` | Components and port connections |
| `input/model-libraries/` | Copy of `antares_legacy_models.yml` |
| `input/data-series/` | Hourly timeseries as plain-text files |

The `lib_paths` argument points to the shared GEMS library shipped with this repository
(`libraries/antares_legacy_models.yml`).

In [ ]:
from antares_gems_converter.input_converter.src.converter import AntaresStudyConverter
from antares_gems_converter.input_converter.src.logger import Logger

# Path to this repository's shared Antares legacy model library
lib_path = Path("../../../libraries/antares_legacy_models.yml").resolve()

print("CONVERTING TO GEMS FORMAT")
print(f"\tUsing library : {lib_path}")

logger = Logger("legacy_tutorial", None)

converter = AntaresStudyConverter(
    study_input=study,
    logger=logger,
    mode="full",
    output_folder=gems_output_dir,
    lib_paths=[str(lib_path)],
)
converter.process_all()

print(f"\t\u2713 system.yml       \u2192 {gems_output_dir / 'input' / 'system.yml'}")
print(f"\t\u2713 model-libraries  \u2192 {gems_output_dir / 'input' / 'model-libraries'}")
print(f"\t\u2713 data-series      \u2192 {gems_output_dir / 'input' / 'data-series'}")

## 3. Inspect the Generated GEMS Study

Let's read `system.yml` to verify that all components and connections were generated
correctly, then save the component IDs by model type for use in the visualisation.

In [ ]:
import yaml
from collections import defaultdict

system_yml_path = gems_output_dir / "input" / "system.yml"
system_data = yaml.safe_load(system_yml_path.read_text())["system"]

components_list = system_data.get("components", [])
connections_list = system_data.get("connections", [])

# Group component IDs by model type
by_model: dict = defaultdict(list)
for comp in components_list:
    by_model[comp["model"]].append(comp["id"])

print(f"System: {system_data['id']}")
print(f"\nComponents ({len(components_list)} total)")
print("-" * 30)
for model, ids in sorted(by_model.items()):
    short = model.split(".")[-1]
    print(f"  [{short}]  ({len(ids)})")
    for cid in ids:
        print(f"    - {cid}")

print(f"\nConnections ({len(connections_list)} total)")
print("-" * 30)
for conn in connections_list:
    print(
        f"  {conn.get('component1')}.{conn.get('port1')}"
        f"  \u2192  {conn.get('component2')}.{conn.get('port2')}"
    )

# Collect IDs by model type — used in Section 5 visualisation
area_ids      = by_model.get("antares_legacy_models.area", [])
thermal_ids   = by_model.get("antares_legacy_models.thermal", [])
renewable_ids = by_model.get("antares_legacy_models.renewable", [])

## 4. Run the Optimisation with GemsPy

[`load_study`](https://gemspy.readthedocs.io/en/latest/) reads `system.yml`, the model
libraries, and the data-series in one call, then returns a `Study` object ready for
solving.

We solve a **one-week, one-scenario** dispatch problem.  
Because both areas have a high ENS cost (20 000 €/MWh), the solver will always prefer
running available generation over leaving demand unmet.

In [ ]:
from gems.study.folder import load_study
from gems.session.session import SimulationSession
from gems.optim_config.parsing import OptimConfig, TimeScopeConfig, ScenarioScopeConfig

TIME_STEPS = 168  # one week

print("LOADING GEMS STUDY")
study_gems = load_study(gems_output_dir)
print("\t\u2713 Study loaded")

optim_config = OptimConfig(
    time_scope=TimeScopeConfig(first_time_step=0, last_time_step=TIME_STEPS - 1),
    scenario_scope=ScenarioScopeConfig(),
)
print("\t\u2713 Optimisation config: 168 time steps, 1 scenario")

print("\nSOLVING OPTIMISATION PROBLEM")
results = SimulationSession(study=study_gems, optim_config=optim_config).run()
print("\t\u2713 Optimisation problem solved")

objective_value = results.data.loc[
    results.data["output"] == "objective-value", "value"
].iloc[0]
print(f"\t\u2713 Objective value: {objective_value:,.0f} \u20ac")

## 5. Visualise Dispatch Results

Two subplots — one per area — show how generation assets are dispatched over the
168-hour horizon:

- **North** — thermal (gas) generation stacked with wind generation; unsupplied energy
  in red when thermal + wind cannot cover the 120 MW load
- **South** — thermal (gas) generation; unsupplied energy in red when the 150 MW
  thermal cannot cover the 80 MW load

Component IDs are discovered dynamically from the inspection step so the plot works
regardless of the exact naming convention used by the converter.

In [ ]:
import matplotlib.pyplot as plt
import io, base64
from IPython.display import display

# --- Discover component IDs by area from the inspection step ---
north_area_id    = next((cid for cid in area_ids      if "north" in cid.lower()), None)
south_area_id    = next((cid for cid in area_ids      if "south" in cid.lower()), None)
north_thermal_id = next((cid for cid in thermal_ids   if "north" in cid.lower()), None)
south_thermal_id = next((cid for cid in thermal_ids   if "south" in cid.lower()), None)
wind_id          = next((cid for cid in renewable_ids if "wind"  in cid.lower()), None)

timesteps = list(range(TIME_STEPS))

def get_output(component_id, output_name, fallback_length=TIME_STEPS):
    """Return time-series values, or zeros if component is not available."""
    if component_id is None:
        return np.zeros(fallback_length)
    return (
        results.component(component_id)
        .output(output_name)
        .value(scenario_index=0)
        .values[:fallback_length]
    )

# Fetch time-series for both areas
thermal_north = get_output(north_thermal_id, "generation_power")
wind_gen      = get_output(wind_id,          "generation_power")
ens_north     = get_output(north_area_id,    "unsupplied_energy")

thermal_south = get_output(south_thermal_id, "generation_power")
ens_south     = get_output(south_area_id,    "unsupplied_energy")

# --- Plot ---
fig, (ax_north, ax_south) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# North area
ax_north.stackplot(
    timesteps, thermal_north, wind_gen,
    labels=["gas_north (100 MW)", "wind (60 MW)"],
    colors=["steelblue", "limegreen"],
    alpha=0.85,
)
ax_north.plot(timesteps, ens_north, color="red",   linewidth=1.5, label="Unsupplied energy")
ax_north.axhline(120,    color="black", linestyle="--", linewidth=1.2, label="Load (120 MW)")
ax_north.set_title("North area")
ax_north.set_ylabel("Power (MW)")
ax_north.legend(loc="upper right", fontsize=8)
ax_north.grid(True, alpha=0.3, axis="y")

# South area
ax_south.stackplot(
    timesteps, thermal_south,
    labels=["gas_south (150 MW)"],
    colors=["darkorange"],
    alpha=0.85,
)
ax_south.plot(timesteps, ens_south, color="red",   linewidth=1.5, label="Unsupplied energy")
ax_south.axhline(80,     color="black", linestyle="--", linewidth=1.2, label="Load (80 MW)")
ax_south.set_title("South area")
ax_south.set_xlabel("Time (hours)")
ax_south.set_ylabel("Power (MW)")
ax_south.legend(loc="upper right", fontsize=8)
ax_south.grid(True, alpha=0.3, axis="y")

fig.suptitle(
    "Generation dispatch — Antares Legacy \u2192 GEMS (1 week, 1 scenario)",
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()

buf = io.BytesIO()
fig.savefig(buf, format="png", bbox_inches="tight")
buf.seek(0)
display(
    {"image/png": base64.b64encode(buf.read()).decode()},
    raw=True,
    metadata={
        "image/png": {
            "alt": (
                "Two subplots over 168 hours. "
                "Top (north area): stacked gas and wind generation with a dashed load line "
                "at 120 MW; red line shows unsupplied energy when gas + wind cannot cover load. "
                "Bottom (south area): gas generation with a dashed load line at 80 MW and "
                "red unsupplied energy when the thermal plant cannot meet demand."
            )
        }
    },
)
plt.close(fig)